In [ ]:
# === ComfyUI + LTX-Video 终极一键完整部署包 (Pinggy 60分钟免注册稳定版) ===
import os
import subprocess
import threading
import time
import re

# 0. 关键修复：确保每次运行前先回到根目录，防止找不到文件夹报错
os.chdir('/content')

# 1. 彻底清理旧文件（防止冲突）
!rm -rf /content/ComfyUI

print("🚀 [1/6] 正在安装 ComfyUI 基础框架...")
!git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI > /dev/null 2>&1
!pip install -q torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu121
!pip install -q -r /content/ComfyUI/requirements.txt

print("📦 [2/6] 正在安装 ComfyUI Manager (插件管家)...")
!git clone https://github.com/ltdrdata/ComfyUI-Manager.git /content/ComfyUI/custom_nodes/ComfyUI-Manager > /dev/null 2>&1

print("🎬 [3/6] 正在安装 LTX-Video 视频专属节点...")
!git clone https://github.com/Lightricks/ComfyUI-LTXVideo.git /content/ComfyUI/custom_nodes/ComfyUI-LTXVideo > /dev/null 2>&1
!pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-LTXVideo/requirements.txt

print("⏳ [4/6] 正在下载 LTX 视频主模型 (约9GB，请稍候)...")
!apt -y install -qq aria2 > /dev/null 2>&1
!mkdir -p /content/ComfyUI/models/checkpoints
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/Lightricks/LTX-Video/resolve/main/ltx-video-2b-v0.9.1.safetensors -d /content/ComfyUI/models/checkpoints -o ltx-video-2b-v0.9.1.safetensors > /dev/null 2>&1

print("🧠 [5/6] 正在下载 T5 文本翻译官模型 (约4GB，请稍候)...")
!mkdir -p /content/ComfyUI/models/clip
!aria2c --console-log-level=error -c -x 16 -s 16 -k 1M https://huggingface.co/comfyanonymous/flux_text_encoders/resolve/main/t5xxl_fp8_e4m3fn.safetensors -d /content/ComfyUI/models/clip -o t5xxl_fp8_e4m3fn.safetensors > /dev/null 2>&1

print("🌐 [6/6] 所有零件拼装完毕！正在启动网页服务...")

def start_pinggy(port):
    print("\n" + "="*60)
    print("✅ 正在启动 Pinggy 隧道...")
    print("⏳ 请耐心等待 10 秒左右，会自动抓取专属访问链接...")
    print("="*60 + "\n")

    # 运行 Pinggy 的 SSH 命令进行内网穿透
    command = f"ssh -p 443 -R0:localhost:{port} -o StrictHostKeyChecking=no a.pinggy.io"
    p = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)

    # 实时读取输出，抓取生成的 https 链接
    for line in p.stdout:
        text = line.decode('utf-8', errors='ignore')
        if "https://" in text and "pinggy" in text:
            match = re.search(r'(https?://[^\s]+)', text)
            if match:
                print("\n🎉 成功！请点击下方专属直达链接进入 ComfyUI：")
                print(f"👉 {match.group(1)} 👈")
                print("⚠️ 注意：此免费链接有效期为 60 分钟，超时后重新运行此代码块即可获取新链接。\n")
                break # 抓到链接后退出循环

# 开启隧道线程
threading.Thread(target=start_pinggy, daemon=True, args=(8188,)).start()

# 启动 ComfyUI 主程序
%cd /content/ComfyUI
!python main.py